In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.utilities import SQLDatabase
db = SQLDatabase.from_uri("sqlite:///../resources/Chinook.db")

In [3]:
from langchain.tools import tool


@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [4]:
SYSTEM = f"""You are a careful SQLite analyst of chinook database. Your name is cooper.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows of output unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *."""

In [5]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.5
)

summarization_model = init_chat_model(
    model="openai/gpt-oss-20b",
    model_provider="groq",
    temperature=0.2
)

In [6]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 32768,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    tools=[execute_sql],
    system_prompt=SYSTEM,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            trigger=("tokens", 500),
            keep=("messages", 1)
        )
    ],
)

config = {"configurable": {"thread_id": "1"}}

In [8]:
question = "Which table has the highest number of entries?"

response = agent.invoke(
    {"messages": question},
    config=config
    )
print(response["messages"][-1].content)

The table with the most rows in the Chinook database is **`PlaylistTrack`**, which contains **8,715** records.
